# RF Complexity Analysis (EU)
## Data Loading

We use the `forest_report.json` file in the repo and the unified ETL cache system.

In [ ]:
from pathlib import Path
from etl.loader import etl

RESULTS_DIR = Path("results")
zip_paths = sorted(RESULTS_DIR.glob("*.zip"))

db = etl(
    zip_paths,
    RESULTS_DIR,
    use_cache=True,           # Use the unified cache system
    force_refresh=False,      # Set to True to force refresh
    auto_select=True,
    load_only_db10=True,      # Load ONLY DB10
    verbose=False
)

from etl.tables import (
    prepare_models_analysis,
    print_models_analysis_diagnostics,
)

analysis_context = prepare_models_analysis(db=db, verbose=True, selected_dataset=None)

In [ ]:
from cost_function import cal_sigmas
from etl.reasons_analysis import extract_test_samples

tests_sample, X_test, test_ids, feature_names = extract_test_samples(db)

X_train = db["data"]["TRAINING_SET"]["value_json"]["X_train"]
sigmas_all = cal_sigmas(X_train, X_test, feature_names, test_ids=test_ids)

print(f"Extracted {len(test_ids)} test samples\nFeatures: {len(feature_names)}")

In [ ]:
from cost_function import cost_function
from redis_helpers.icf import bitmap_to_icf
import pandas as pd
from etl.loader import get_etl_cache
from pathlib import Path
from etl.progress import ICFProgressMonitor, CacheWriteCoordinator

print("Calculating costs for anti_reasons...")
print("=" * 80)

cache = get_etl_cache()
dataset_name = db.get('_dataset_name', 'unknown')

# Find the ZIP path for this dataset
RESULTS_DIR = Path("results")
zip_paths = sorted(RESULTS_DIR.glob(f"{dataset_name}_*.zip"))
selected_zip_path = zip_paths[0] if zip_paths else None

# Define all reason types to process
reason_types = ['reasons', 'non_reasons', 'anti_reasons']

# Try to load from cache
cached_data = None
if selected_zip_path:
    cached_data = cache.load_costs(selected_zip_path, reason_types)

if cached_data is not None:
    print(f"\n Loaded costs from cache for {dataset_name}")
    print("=" * 80)
    cost_df = cached_data['cost_df']
    # Update tests_sample with cached data
    for sample_id in test_ids:
        if sample_id in cached_data['tests_sample']:
            for reason_type in reason_types:
                if reason_type in cached_data['tests_sample'][sample_id]:
                    tests_sample[sample_id][reason_type] = cached_data['tests_sample'][sample_id][reason_type]
else:
    print(f"\n→ Computing costs for {dataset_name} (not in cache)...")
    print("Using PARALLEL computation with INCREMENTAL cache to reduce RAM usage")

    # Import parallel cost calculation
    from etl.parallel_costs import calculate_costs_parallel_incremental

    progress_monitor = ICFProgressMonitor(
        project_name=f"{dataset_name} ICF",
        refresh_interval=1.0,
        enabled=True
    )

    with CacheWriteCoordinator(cache=cache, zip_path=selected_zip_path, verbose=False) as cache_writer:
        # Calculate costs in parallel with incremental cache saves
        total_costs, tests_sample = calculate_costs_parallel_incremental(
            db=db,
            test_ids=test_ids,
            tests_sample=tests_sample,
            sigmas_all=sigmas_all,
            cost_function=cost_function,
            bitmap_to_icf=bitmap_to_icf,
            reason_types=reason_types,
            cache=cache,
            selected_zip_path=selected_zip_path,
            n_workers=None,  # Auto-detect CPU count
            batch_size=50,    # Process 50 ICFs per batch
            save_every_n_batches=10,  # Save to cache every 10 batches
            verbose=True,
            progress_monitor=progress_monitor,
            cache_writer=cache_writer
        )

    print(f"\n Parallel computation complete: {total_costs} costs calculated")

    # Load cost_df from cache (already saved incrementally)
    cached_costs = cache.load_costs(selected_zip_path, reason_types)
    if cached_costs:
        cost_df = cached_costs['cost_df']
    else:
        # Fallback: create empty DataFrame
        cost_df = pd.DataFrame()


print(f"\n{'='*80}")
print(f"COST CALCULATION COMPLETE")
print(f"{'='*80}")
print(f"Total costs calculated: {len(cost_df)}")


In [ ]:
from etl.reasons_analysis import calculate_robustness_per_bitmap, calculate_sample_robustness

num_features = len(feature_names)

print(f"\n\n{'='*80}")
print(f"  ROBUSTNESS CALCULATION (Anti-Reasons Only)")
print(f"{'='*80}\n")
print(f"Configuration:")
print(f"   - Number of features: {num_features}")
print(f"\nFormula:")
print(f"   r(C,x) = 1 - max{{ICF ∈ AR{{C,y}}}} cost_x(ICF) / |features|")
print(f"\n   where AR{{C,y}} = set of Anti-Reasons for class y")

# Check if anti_reasons exist in cost_df
if len(cost_df) == 0 or 'reason_type' not in cost_df.columns:
    print("\n ERROR: Cannot calculate robustness - no cost data available.")
    print("  Please ensure the database contains anti_reasons data.")
    print("  Skipping robustness calculation section.")
else:
    # Filter only anti_reasons for robustness calculation
    anti_reasons_df = cost_df[cost_df['reason_type'] == 'anti_reasons'].copy()

    if len(anti_reasons_df) == 0:
        print("\nWARNING: No anti_reasons found in cost data.")
    else:
        # Calculate robustness for each anti-reason ICF
        # This finds the maximum cost across all samples for each anti-reason
        bitmap_robustness = calculate_robustness_per_bitmap(anti_reasons_df, num_features=num_features)

        print(f"\n\n{'='*80}")
        print(f"  ICF-LEVEL ROBUSTNESS ANALYSIS (Anti-Reasons)")
        print(f"{'='*80}\n")
        print(f"ICF Summary:")
        print(f"   - Total anti-reason ICFs: {len(bitmap_robustness)}")
        print(f"\nCost Statistics:")
        print(f"   - Max cost:      {bitmap_robustness['max_cost'].max():.6f}")
        print(f"   - Mean max cost: {bitmap_robustness['max_cost'].mean():.6f}")
        print(f"   - Min max cost:  {bitmap_robustness['max_cost'].min():.6f}")
        print(f"\nRobustness Statistics:")
        print(f"   - Max:  {bitmap_robustness['robustness'].max():.6f}")
        print(f"   - Mean: {bitmap_robustness['robustness'].mean():.6f}")
        print(f"   - Min:  {bitmap_robustness['robustness'].min():.6f}")

        # Use ETL function to calculate robustness for all samples
        from etl.reasons_analysis import calculate_all_samples_robustness, print_robustness_statistics

        sample_robustness_df = calculate_all_samples_robustness(
            cost_df=cost_df,
            num_features=num_features,
            tests_sample=tests_sample,
            test_ids=test_ids,
            verbose=True
        )

        # Print comprehensive statistics
        print_robustness_statistics(sample_robustness_df)

        # Save results
        sample_robustness_df.to_csv(f'results/{dataset_name}_sample_robustness.csv', index=False)
        print(f"\nResults saved to: results/{dataset_name}_sample_robustness.csv\n")

In [ ]:
from etl.tables import build_accuracy_vs_robustness_report

sample_robustness_ref = locals().get('sample_robustness_df')
report = build_accuracy_vs_robustness_report(db, dataset_name, sample_robustness_ref)

for line in report['lines']:
    print(line)

In [ ]:
# Statistical Visualizations for Sample Robustness
from etl.reasons_analysis import create_robustness_visualizations

if 'sample_robustness_df' in locals() and len(sample_robustness_df) > 0:
    # Create visualizations using ETL function
    fig_main, fig_quartiles = create_robustness_visualizations(
        sample_robustness_df=sample_robustness_df,
        dataset_name=dataset_name
    )

    if fig_main is not None:
        fig_main.show()

    if fig_quartiles is not None:
        fig_quartiles.show()
else:
    print("sample_robustness_df not available. Run previous cells first.")


In [ ]:
# Visualize actual ICFs from the dataset
from etl.reasons_analysis import visualize_all_time_series

print(f"Total test samples available: {len(test_ids)}")
print(f"Number of features per sample: {len(feature_names)}")

# Visualize all time series (max 50 samples)
fig = visualize_all_time_series(tests_sample, test_ids, feature_names, max_samples=50)
fig.show()

# Select first sample for detailed analysis
sample_id = test_ids[0]

print(f"\n{'='*80}")
print(f"Detailed analysis will use Sample ID: {sample_id}")
print(f"{'='*80}")


In [ ]:
# Plot 2: Time series with REASON (maximal reason from dataset)
from etl.reasons_analysis import visualize_sample_with_icf

fig = visualize_sample_with_icf(sample_id, tests_sample, feature_names, reason_type='reasons')
if fig is not None:
    fig.show()


In [ ]:
# Plot 3: Time series with ANTI-REASON
fig = visualize_sample_with_icf(sample_id, tests_sample, feature_names, reason_type='anti_reasons')
if fig is not None:
    fig.show()


In [ ]:
# Plot 4: Combined view - Reason vs Anti-Reason
from etl.reasons_analysis import visualize_sample_comparison

fig = visualize_sample_comparison(sample_id, tests_sample, feature_names)
if fig is not None:
    fig.show()


In [ ]:
# Calculate robustness per bitmap (ICF) - ONLY for Anti-Reasons
from etl.reasons_analysis import calculate_robustness_per_bitmap
print("\n" + "=" * 80)
# Filter only anti-reasons
print("ROBUSTNESS PER ICF (Anti-Reasons Only)")
print("=" * 80)
if len(anti_reasons_df) == 0:
    print("\nNo anti-reasons found in cost data.")
else:
    # Calculate with normalization
    bitmap_robustness = calculate_robustness_per_bitmap(anti_reasons_df, num_features=num_features)

    print(f"\nTotal anti-reason ICFs analyzed: {len(bitmap_robustness)}")
    print(f"\nTop 10 ICFs by max_cost (hardest to reach):")
    print(bitmap_robustness[['max_cost', 'robustness', 'mean_cost', 'n_samples']].head(10).to_string(index=False))

    # Anti-reasons with lowest robustness (easiest to change classification)
    print(f"\n\nANTI-REASONS with LOWEST robustness (most vulnerable - easiest to perturb):")
    ar_lowest = bitmap_robustness.nsmallest(5, 'robustness')
    print(ar_lowest[['max_cost', 'robustness', 'mean_cost', 'n_samples']].to_string(index=False))

    # Save results (including bitmap_index for reference, but not displayed)
    bitmap_robustness.to_csv(f'results/{dataset_name}_anti_reasons_robustness.csv', index=False)
    print(f"\n\nResults saved to: results/anti_reasons_robustness.csv")

In [ ]:
# Plot 4: Combined view - Reason vs Anti-Reason
from etl.reasons_analysis import visualize_sample_comparison_smooth

fig = visualize_sample_comparison_smooth(sample_id, tests_sample, feature_names)
if fig is not None:
    fig.show()

In [ ]:
from etl.reasons_analysis import visualize_anti_reason_corridor

fig = visualize_anti_reason_corridor(sample_id, tests_sample, feature_names)
if fig is not None:
    fig.show()

In [ ]:
analysis_context.first_table.summary_styler

In [ ]:
print_models_analysis_diagnostics(analysis_context)
analysis_context.summary_styler

In [ ]:
# Display the combined analyzed table
print("Combined Analyzed Results (Test Accuracy & Performance Metrics):")
print("="*80)
analysis_context.combined_analyzed_styler

In [ ]:
# Save the combined analyzed data to CSV
combined_df = analysis_context.combined_analyzed_styler.data
combined_df.to_csv("combined_analyzed_results.csv")

print(f"Saved combined analysis results to: combined_analyzed_results.csv")
print(f"Shape: {combined_df.shape}")
print(f"Columns: {list(combined_df.columns)}")
print(f"Index: {list(combined_df.index)}")

# Also save the first table summary
summary_df = analysis_context.first_table.summary_styler.data
summary_df.to_csv("summary_results.csv", index=False)

print(f"Saved summary results to: summary_results.csv")
print(f"Shape: {summary_df.shape}")
print(f"Columns: {list(summary_df.columns)}")

# Save the analyzed counts
counts_df = analysis_context.analyzed_counts_df
counts_df.to_csv("analyzed_counts.csv", index=False)

print(f"Saved analyzed counts to: analyzed_counts.csv") 
print(f"Shape: {counts_df.shape}")
print(f"Columns: {list(counts_df.columns)}")

In [ ]:
import pandas as pd
combined_df = pd.read_csv("combined_analyzed_results.csv")

In [ ]:
combined_df

In [ ]:
# Combine Mean EU Features and EU Std into a single row, then add data from other CSVs
import pandas as pd
import numpy as np

# Read the current combined data
combined_df = pd.read_csv("combined_analyzed_results.csv", index_col=0)

# Check if both rows exist
if 'Mean EU Features' in combined_df.index and 'EU Std' in combined_df.index:
    # Get the data for both rows
    mean_row = combined_df.loc['Mean EU Features']
    std_row = combined_df.loc['EU Std']
    
    # Create the combined row with format "mean (± std)"
    combined_row = pd.Series(index=combined_df.columns, dtype=object)
    
    for col in combined_df.columns:
        mean_val = mean_row[col]
        std_val = std_row[col]
        
        # Check if both values are valid numbers
        if pd.notna(mean_val) and pd.notna(std_val):
            try:
                mean_num = float(mean_val)
                std_num = float(std_val)
                combined_row[col] = f"{mean_num:.3f} (± {std_num:.3f})"
            except (ValueError, TypeError):
                combined_row[col] = str(mean_val)  # fallback to original mean value
        elif pd.notna(mean_val):
            combined_row[col] = str(mean_val)
        else:
            combined_row[col] = ""
    
    # Remove the original rows and add the combined row
    combined_df = combined_df.drop(['Mean EU Features', 'EU Std'])
    
    # Insert the combined row at the position where Mean EU Features was
    # Find the best position (after N Estimators, before Test Accuracy if it exists)
    insert_idx = 4  # Default position
    if 'N Estimators' in combined_df.index:
        insert_idx = list(combined_df.index).index('N Estimators') + 1
    
    # Insert the new row
    combined_df_list = combined_df.index.tolist()
    combined_df_list.insert(insert_idx, 'Mean EU Features (± Std)')
    
    # Reindex with the new order
    new_combined_df = pd.DataFrame(index=combined_df_list, columns=combined_df.columns)
    for idx in combined_df.index:
        new_combined_df.loc[idx] = combined_df.loc[idx]
    new_combined_df.loc['Mean EU Features (± Std)'] = combined_row
    
    combined_df = new_combined_df
    
    print("Successfully combined Mean EU Features and EU Std")

# Now add data from summary_results.csv and analyzed_counts.csv
try:
    # Load summary results
    summary_df = pd.read_csv("summary_results.csv")
    print(f"Loaded summary_results.csv with {summary_df.shape[0]} datasets")
    
    # Load analyzed counts
    counts_df = pd.read_csv("analyzed_counts.csv")
    print(f"Loaded analyzed_counts.csv with {counts_df.shape[0]} datasets")
    
    # Create additional rows for summary data that's not already in combined_df
    summary_metrics_to_add = ['n_features','eu_complexity','eu_min','eu_max']
    
    
    # Add summary metrics as new rows
    for metric in summary_metrics_to_add:
        if metric in summary_df.columns:
            new_row = pd.Series(index=combined_df.columns, dtype=object)
            
            # Map dataset names to values
            for _, row in summary_df.iterrows():
                dataset = str(row['dataset'])
                if dataset in combined_df.columns:
                    value = row[metric]
                    if pd.notna(value):
                        new_row[dataset] = value
            
            # Add the new row
            metric_name = metric.replace('_', ' ').title()
            if metric == 'n_features':
                metric_name = 'N Features'
            elif metric == 'eu_complexity':
                metric_name = 'EU Complexity'
            elif metric == 'eu_min':
                metric_name = 'EU Min'
            elif metric == 'eu_max':
                metric_name = 'EU Max'
                
            combined_df.loc[metric_name] = new_row
            print(f"Added {metric_name} from summary_results.csv")
    
    # Add metrics from analyzed_counts that aren't already present
    counts_metrics_to_add = [
        'selected_sample',  # If this column exists
    ]
    
    # Check what additional metrics are in counts_df
    for col in counts_df.columns:
        if col != 'dataset' and col not in ['Total time (s) max', 'Total time (s) mean', 
                                           'ICF checks', 'Reason check iteration total',
                                           'IterGoodRatio', 'IterBadRatio', 'Early Stop Good total',
                                           'Early Stop from Good', 'Early Stop from Bad', 'Filtrered rate']:
            counts_metrics_to_add.append(col)
    
    # Add counts metrics as new rows
    for metric in counts_metrics_to_add:
        if metric in counts_df.columns:
            new_row = pd.Series(index=combined_df.columns, dtype=object)
            
            # Map dataset names to values
            for _, row in counts_df.iterrows():
                dataset = str(row['dataset'])
                if dataset in combined_df.columns and dataset != 'All workers':
                    value = row[metric]
                    if pd.notna(value):
                        new_row[dataset] = value
            
            # Add the new row if it has any data
            if new_row.notna().any():
                metric_name = metric.replace('_', ' ').title()
                combined_df.loc[metric_name] = new_row
                print(f"Added {metric_name} from analyzed_counts.csv")
    
    print(f"\nFinal combined dataframe shape: {combined_df.shape}")
    print(f"Total metrics: {len(combined_df.index)}")
    print(f"Datasets: {len(combined_df.columns)}")
    
except Exception as e:
    print(f"Error adding data from CSV files: {e}")

# Display the updated dataframe
combined_df.to_csv("combined_analyzed_results_updated.csv")
combined_df

In [ ]:
# Add robustness metrics from {dataset_name}_accuracy_vs_robustness_report.json files
import json
import os
from pathlib import Path

print(f"\nAdding robustness metrics from JSON reports...")
print("=" * 60)

# Define the robustness metrics to extract
robustness_metrics = ['mean', 'min', 'max']

# Initialize new rows for robustness metrics
for metric in robustness_metrics:
    metric_name = f'Robustness {metric.title()}'
    combined_df.loc[metric_name] = pd.Series(index=combined_df.columns, dtype=object)

# Look for JSON report files in the results directory
results_dir = Path("results")
json_files = list(results_dir.glob("*_accuracy_vs_robustness_report.json"))

print(f"Found {len(json_files)} robustness report files")

for json_file in json_files:
    try:
        # Extract dataset name from filename
        # Format: {dataset_name}_accuracy_vs_robustness_report.json
        filename = json_file.stem  # Remove .json extension
        dataset_name = filename.replace('_accuracy_vs_robustness_report', '')
        
        # Check if this dataset exists in our combined dataframe
        if dataset_name in combined_df.columns:
            # Load the JSON report
            with open(json_file, 'r') as f:
                report = json.load(f)
            
            # Extract robustness statistics
            if 'robustness_stats' in report:
                robustness_stats = report['robustness_stats']
                
                # Parse the robustness stats string
                # Format: "mean      0.609465\nmedian    0.610535\nmin       0.462442\nmax       0.690457\nName: robustness, dtype: float64"
                if isinstance(robustness_stats, str):
                    lines = robustness_stats.split('\n')
                    stats_dict = {}
                    
                    for line in lines:
                        if '    ' in line and not line.startswith('Name:'):
                            parts = line.split()
                            if len(parts) >= 2:
                                stat_name = parts[0]
                                stat_value = parts[1]
                                try:
                                    stats_dict[stat_name] = float(stat_value)
                                except ValueError:
                                    continue
                    
                    # Add the extracted metrics to the combined dataframe
                    for metric in robustness_metrics:
                        if metric in stats_dict:
                            metric_name = f'Robustness {metric.title()}'
                            combined_df.loc[metric_name, dataset_name] = round(stats_dict[metric], 6)
                    
                    print(f"Added robustness metrics for {dataset_name}")
                    print(f"   Mean: {stats_dict.get('mean', 'N/A'):.6f}, Min: {stats_dict.get('min', 'N/A'):.6f}, Max: {stats_dict.get('max', 'N/A'):.6f}")
                
                elif isinstance(robustness_stats, dict):
                    # Handle case where robustness_stats is already a dictionary
                    for metric in robustness_metrics:
                        if metric in robustness_stats:
                            metric_name = f'Robustness {metric.title()}'
                            combined_df.loc[metric_name, dataset_name] = round(robustness_stats[metric], 6)
                    
                    print(f"Added robustness metrics for {dataset_name} (dict format)")
            else:
                print(f"No robustness_stats found in {json_file.name}")
        else:
            print(f"Dataset '{dataset_name}' not found in combined dataframe columns")
            
    except Exception as e:
        print(f"Error processing {json_file.name}: {e}")

# Count how many datasets got robustness data
robustness_data_count = 0
for metric in robustness_metrics:
    metric_name = f'Robustness {metric.title()}'
    non_null_count = combined_df.loc[metric_name].notna().sum()
    robustness_data_count = max(robustness_data_count, non_null_count)

print(f"\nSuccessfully added robustness metrics for {robustness_data_count} datasets")
print(f"Added metrics: {[f'Robustness {m.title()}' for m in robustness_metrics]}")

# Save the updated dataframe
combined_df.to_csv("combined_analyzed_results_updated.csv")
print(f"\nUpdated combined_analyzed_results_updated.csv with robustness metrics")

In [ ]:
# Display the final updated dataframe with robustness metrics
print("Final Combined Analysis Results with Robustness Metrics:")
print("=" * 70)
print(f"Shape: {combined_df.shape}")
print(f"Total metrics: {len(combined_df.index)}")
print(f"Datasets: {len(combined_df.columns)}")

# Show which metrics are now included
print(f"\nAvailable metrics:")
for idx in combined_df.index:
    print(f"   - {idx}")

# Display the dataframe
combined_df

In [ ]:
# Add Redis reason counts to each dataset
print("\nAdding Redis reason counts...")
print("=" * 50)

try:
    # Read the redis reason counts CSV file
    redis_counts_df = pd.read_csv("results/redis_reason_counts.csv")
    print(f"Loaded redis_reason_counts.csv with {redis_counts_df.shape[0]} datasets")
    
    # Define which redis metrics to add as separate rows
    redis_metrics_to_add = [
        'Candidate',
        'Reason', 
        'Non-reason',
        'Candidate Anti-reason',
        'Anti-reason',
        'Good profile',
        'Bad profile',
        'Preferred reason',
        'Anti-reason profile',
        'Total'
    ]
    
    # Add redis metrics as new rows
    for metric in redis_metrics_to_add:
        if metric in redis_counts_df.columns:
            new_row = pd.Series(index=combined_df.columns, dtype=object)
            
            # Map dataset names to values
            for _, row in redis_counts_df.iterrows():
                dataset = str(row['dataset'])
                if dataset in combined_df.columns:
                    value = row[metric]
                    if pd.notna(value):
                        new_row[dataset] = int(value)  # Convert to int for count data
            
            # Add the new row with Redis prefix to distinguish from other metrics
            metric_name = f'{metric}'
            combined_df.loc[metric_name] = new_row
            
            # Count non-null values
            non_null_count = new_row.notna().sum()
            print(f"Added {metric_name} - {non_null_count} datasets have data")
    
    # Save the updated dataframe
    combined_df.to_csv("combined_analyzed_results_updated.csv")
    print(f"\nUpdated combined_analyzed_results_updated.csv with Redis reason counts")
    print(f"Total Redis metrics added: {len(redis_metrics_to_add)}")

except Exception as e:
    print(f"Error adding Redis reason counts: {e}")
    print("   Make sure results/redis_reason_counts.csv exists and is readable")

In [ ]:
combined_df.to_csv("combined_analyzed_results_updated.csv")

In [ ]:
#!/usr/bin/env python3
"""
Update combined_analyzed_results_updated.csv to include standard deviations 
with the robustness means in the format "mean (± std)"
"""

import pandas as pd
import numpy as np

# Load the datasets
print("Loading datasets...")
combined_df = pd.read_csv("combined_analyzed_results_updated.csv", index_col=0)
individual_stats_df = pd.read_csv("results/individual_dataset_robustness_stats.csv")

print(f"Combined dataframe shape: {combined_df.shape}")
print(f"Individual stats shape: {individual_stats_df.shape}")

# Create a mapping of dataset -> robustness std for the "Overall" category
robustness_std_map = {}
overall_stats = individual_stats_df[individual_stats_df['category'] == 'Overall']

for _, row in overall_stats.iterrows():
    dataset = row['dataset']
    std_value = row['std']
    robustness_std_map[dataset] = std_value

print(f"\nFound robustness std data for {len(robustness_std_map)} datasets:")
for dataset, std_val in robustness_std_map.items():
    print(f"  {dataset}: {std_val:.6f}")

# Update the Robustness Mean row to include standard deviations
if 'Robustness Mean' in combined_df.index:
    print("\nUpdating Robustness Mean with standard deviations...")
    
    # Create a new row with mean ± std format
    new_robustness_row = pd.Series(index=combined_df.columns, dtype=object)
    
    for dataset in combined_df.columns:
        mean_value = combined_df.loc['Robustness Mean', dataset]
        
        if pd.notna(mean_value) and dataset in robustness_std_map:
            mean_float = float(mean_value)
            std_float = robustness_std_map[dataset]
            new_robustness_row[dataset] = f"{mean_float:.6f} (± {std_float:.6f})"
        elif pd.notna(mean_value):
            # Keep the original value if no std data available
            new_robustness_row[dataset] = f"{float(mean_value):.6f}"
        else:
            # Keep empty if no mean data
            new_robustness_row[dataset] = ""
    
    # Replace the old row with the new formatted one
    combined_df.loc['Robustness Mean (± Std)'] = new_robustness_row
    
    # Remove the old Robustness Mean row
    combined_df = combined_df.drop('Robustness Mean')
    
    print("Successfully updated Robustness Mean with standard deviations")
    
    # Show some examples of the updated format
    print("\nExample updated values:")
    for i, (dataset, value) in enumerate(new_robustness_row.items()):
        if value and i < 5:  # Show first 5 non-empty values
            print(f"  {dataset}: {value}")
else:
    print("'Robustness Mean' row not found in combined dataframe")

# Save the updated dataframe
output_file = "combined_analyzed_results_updated.csv"
combined_df.to_csv(output_file)
print(f"\nSaved updated file: {output_file}")

# Display the updated dataframe structure
print(f"\nUpdated dataframe shape: {combined_df.shape}")
print(f"Metrics now include:")

# Show which robustness-related metrics are now present
robustness_metrics = [idx for idx in combined_df.index if 'Robustness' in idx]
for metric in robustness_metrics:
    print(f"  - {metric}")

print("\nSummary:")
print(f"   - Updated {len([col for col in combined_df.columns if pd.notna(combined_df.loc['Robustness Mean (± Std)', col]) and '±' in str(combined_df.loc['Robustness Mean (± Std)', col])])} datasets with mean ± std format")
print(f"   - Robustness means now include standard deviations where available")

In [ ]:
import pandas as pd
df = pd.read_csv("combined_analyzed_results_updated.csv")
df

In [17]:
# Create individual boxplots for each dataset showing robustness by classification accuracy
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas as pd
import json
from pathlib import Path

print("Creating Individual Robustness Boxplots for Each Dataset")
print("=" * 70)

# Load the comprehensive analysis data
df = pd.read_csv("combined_analyzed_results_updated.csv", index_col=0)

# Get all datasets
datasets = df.columns.tolist()
print(f"Found {len(datasets)} datasets to analyze")

# Collect robustness data for all datasets
all_dataset_data = {}
results_dir = Path("results")

for dataset in datasets:
    try:
        # Look for the sample robustness CSV file
        sample_robustness_file = results_dir / f"{dataset}_sample_robustness.csv"
        
        if sample_robustness_file.exists():
            # Load sample robustness data
            sample_df = pd.read_csv(sample_robustness_file)
            
            # Check if required columns exist
            if all(col in sample_df.columns for col in ['robustness', 'correct_prediction']):
                
                # Prepare data for this dataset
                dataset_robustness_data = []
                # Overall samples (all samples)
                for robustness_val in sample_df['robustness']:
                    dataset_robustness_data.append({
                        'robustness': robustness_val,
                        'category': 'Overall',
                        'type': 'All Samples'
                    })
                # Correctly classified samples
                correct_samples = sample_df[sample_df['correct_prediction'] == True]
                for robustness_val in correct_samples['robustness']:
                    dataset_robustness_data.append({
                        'robustness': robustness_val,
                        'category': 'Correct',
                        'type': 'Correctly Classified'
                    })
                
                # Incorrectly classified samples
                incorrect_samples = sample_df[sample_df['correct_prediction'] == False]
                for robustness_val in incorrect_samples['robustness']:
                    dataset_robustness_data.append({
                        'robustness': robustness_val,
                        'category': 'Incorrect', 
                        'type': 'Incorrectly Classified'
                    })
                
                
                
                if dataset_robustness_data:
                    all_dataset_data[dataset] = {
                        'data': pd.DataFrame(dataset_robustness_data),
                        'n_correct': len(correct_samples),
                        'n_incorrect': len(incorrect_samples),
                        'total': len(sample_df)
                    }
                    print(f"Loaded {len(sample_df)} samples for {dataset}")
                    print(f"   Correct: {len(correct_samples)}, Incorrect: {len(incorrect_samples)}")
                else:
                    print(f"No valid robustness data for {dataset}")
            else:
                print(f"Missing required columns in {sample_robustness_file.name}")
        else:
            print(f"Sample robustness file not found: {sample_robustness_file.name}")
            
    except Exception as e:
        print(f"Error processing {dataset}: {e}")

# Create individual plots for each dataset
print(f"\nCreating individual plots for {len(all_dataset_data)} datasets...")
print("=" * 70)

# Store all plots and statistics
all_stats = []

# Calculate adaptive font sizes based on number of datasets
n_datasets = len(all_dataset_data)
if n_datasets <= 5:
    base_font_size = 28
    title_font_size = 36
elif n_datasets <= 10:
    base_font_size = 28
    title_font_size = 28
elif n_datasets <= 20:
    base_font_size = 28
    title_font_size = 26
else:
    base_font_size = 28
    title_font_size = 26

print(f"Using adaptive font sizes: base={base_font_size}, title={title_font_size} for {n_datasets} datasets")

for dataset, dataset_info in all_dataset_data.items():
    robustness_df = dataset_info['data']
    
    if len(robustness_df) > 0:
        # Extract accuracy for this dataset from the comprehensive analysis
        accuracy_score = "N/A"
        if dataset in df.columns and 'Test Accuracy' in df.index:
            try:
                acc_value = df.loc['Test Accuracy', dataset]
                if pd.notna(acc_value):
                    # Handle different accuracy formats (percentage or decimal)
                    if isinstance(acc_value, str) and '%' in acc_value:
                        accuracy_score = acc_value
                    else:
                        accuracy_score = f"{float(acc_value):.3f}"
            except:
                accuracy_score = "N/A"
        
        # Create individual boxplot for this dataset
        fig = px.box(
            robustness_df, 
            y='robustness', 
            x='category',
            color='category',
            title=f'{dataset} (Acc: {accuracy_score})<br>',
            labels={
                'category': 'Classification Result',
                'robustness': 'Robustness Score'
            },
            color_discrete_map={
                'Overall': '#1f77b4',       # Blue
                'Correct': '#2ca02c',      # Green  
                'Incorrect': '#d62728'     # Red
            },
            category_orders={'category': ['Overall', 'Correct', 'Incorrect', ]}
        )
        
        # Update layout with adaptive font sizes
        fig.update_layout(
            width=600,
            height=500,
            xaxis_title="Classification Result",
            yaxis_title="Robustness Score",
            showlegend=False,
            font=dict(size=base_font_size),
            title_font_size=title_font_size,
            margin=dict(t=80, b=60, l=60, r=60)
        )
        
        # Update traces for better visibility
        fig.update_traces(
            boxpoints='outliers',  # Show outliers
            pointpos=0,
            jitter=0.3
        )
        
        # Show the plot
        fig.show()
        if not Path("results/pdf").exists():
            Path("results/pdf").mkdir(parents=True) 
        fig.write_image(f"results/pdf/{dataset}_robustness_boxplot.pdf", scale=2)
        # Calculate and store statistics for this dataset
        stats = robustness_df.groupby('category')['robustness'].agg([
            'count', 'mean', 'std', 'min', 'max'
        ]).round(6)
        
        print(f"\n{dataset} Statistics:")
        for category in ['Correct', 'Incorrect', 'Overall']:
            if category in stats.index:
                stat_row = stats.loc[category]
                all_stats.append({
                    'dataset': dataset,
                    'category': category,
                    'count': int(stat_row['count']),
                    'mean': stat_row['mean'],
                    'std': stat_row['std'],
                    'min': stat_row['min'],
                    'max': stat_row['max']
                })
                print(f"   {category:9s}: μ={stat_row['mean']:.4f} σ={stat_row['std']:.4f} "
                      f"min={stat_row['min']:.4f} max={stat_row['max']:.4f} n={int(stat_row['count'])}")
    
# Create summary comparison across all datasets
if all_stats:
    print(f"\nSUMMARY COMPARISON ACROSS ALL DATASETS")
    print("=" * 70)
    
    stats_df = pd.DataFrame(all_stats)
    
    # Calculate differences between correct and incorrect predictions
    comparison_data = []
    for dataset in stats_df['dataset'].unique():
        dataset_stats = stats_df[stats_df['dataset'] == dataset]
        correct_stats = dataset_stats[dataset_stats['category'] == 'Correct']
        incorrect_stats = dataset_stats[dataset_stats['category'] == 'Incorrect']
        
        if len(correct_stats) > 0 and len(incorrect_stats) > 0:
            correct_mean = correct_stats['mean'].iloc[0]
            incorrect_mean = incorrect_stats['mean'].iloc[0]
            difference = correct_mean - incorrect_mean
            
            comparison_data.append({
                'dataset': dataset,
                'correct_mean': correct_mean,
                'incorrect_mean': incorrect_mean,
                'difference': difference,
                'correct_count': correct_stats['count'].iloc[0],
                'incorrect_count': incorrect_stats['count'].iloc[0]
            })
    
    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)
        comparison_df = comparison_df.sort_values('difference', ascending=False)
        
        print("\nDatasets ranked by robustness difference (Correct - Incorrect):")
        print("Higher positive values = correctly classified samples are more robust")
        print("-" * 80)
        
        for _, row in comparison_df.iterrows():
            print(f"(Correct: {row['correct_mean']:.4f}, Incorrect: {row['incorrect_mean']:.4f})")
            print(f"{row['dataset']:25s}: Δ={row['difference']:+.4f} "
                )
        
        # Save detailed statistics
        stats_df.to_csv("results/individual_dataset_robustness_stats.csv", index=False)
        comparison_df.to_csv("results/robustness_comparison_summary.csv", index=False)
        
        print(f"\nSaved detailed statistics:")
        print(f"   Individual stats: results/individual_dataset_robustness_stats.csv")
        print(f"   Comparison summary: results/robustness_comparison_summary.csv")


else:    print("No robustness data found. Make sure sample robustness files exist.")

Creating Individual Robustness Boxplots for Each Dataset
Found 18 datasets to analyze
Loaded 36 samples for Wine
   Correct: 24, Incorrect: 12
Loaded 110 samples for MiddlePhalanxOutlineCorrect
   Correct: 90, Incorrect: 20
Loaded 163 samples for SonyAIBORobotSurface1
   Correct: 145, Incorrect: 18
Loaded 13 samples for BeetleFly
   Correct: 10, Incorrect: 3
Loaded 655 samples for TwoLeadECG
   Correct: 487, Incorrect: 168
Sample robustness file not found: HandOutlines_sample_robustness.csv
Loaded 26 samples for Lightning2
   Correct: 19, Incorrect: 7
Loaded 15 samples for FaceFour
   Correct: 14, Incorrect: 1
Loaded 94 samples for ToeSegmentation2
   Correct: 84, Incorrect: 10
Loaded 29 samples for ECG200
   Correct: 23, Incorrect: 6
Loaded 495 samples for ItalyPowerDemand
   Correct: 476, Incorrect: 19
Loaded 18 samples for Meat
   Correct: 17, Incorrect: 1
Loaded 445 samples for SonyAIBORobotSurface2
   Correct: 307, Incorrect: 138
Loaded 15 samples for Coffee
   Correct: 15, Incorr


Wine Statistics:
   Correct  : μ=0.7097 σ=0.0127 min=0.6711 max=0.7246 n=24
   Incorrect: μ=0.7017 σ=0.0169 min=0.6754 max=0.7212 n=12
   Overall  : μ=0.7070 σ=0.0145 min=0.6711 max=0.7246 n=36



MiddlePhalanxOutlineCorrect Statistics:
   Correct  : μ=0.7921 σ=0.0373 min=0.6852 max=0.8814 n=90
   Incorrect: μ=0.7890 σ=0.0489 min=0.6516 max=0.8596 n=20
   Overall  : μ=0.7916 σ=0.0394 min=0.6516 max=0.8814 n=110



SonyAIBORobotSurface1 Statistics:
   Correct  : μ=0.7057 σ=0.0053 min=0.6870 max=0.7189 n=145
   Incorrect: μ=0.6977 σ=0.0049 min=0.6903 max=0.7092 n=18
   Overall  : μ=0.7048 σ=0.0058 min=0.6870 max=0.7189 n=163



BeetleFly Statistics:
   Correct  : μ=0.8221 σ=0.0104 min=0.8152 max=0.8496 n=10
   Incorrect: μ=0.8137 σ=0.0163 min=0.7957 max=0.8277 n=3
   Overall  : μ=0.8202 σ=0.0118 min=0.7957 max=0.8496 n=13


...


Coffee Statistics:
   Correct  : μ=0.7577 σ=0.0303 min=0.6877 max=0.7939 n=15
   Overall  : μ=0.7577 σ=0.0303 min=0.6877 max=0.7939 n=15



BirdChicken Statistics:
   Correct  : μ=0.8433 σ=0.0008 min=0.8424 max=0.8443 n=5
   Incorrect: μ=0.8281 σ=0.0166 min=0.7997 max=0.8421 n=5
   Overall  : μ=0.8357 σ=0.0136 min=0.7997 max=0.8443 n=10



GunPoint Statistics:
   Correct  : μ=0.8542 σ=0.0019 min=0.8503 max=0.8594 n=69
   Incorrect: μ=0.8530 σ=0.0016 min=0.8510 max=0.8563 n=12
   Overall  : μ=0.8541 σ=0.0019 min=0.8503 max=0.8594 n=81



CinCECGTorso Statistics:
   Correct  : μ=0.8230 σ=0.0018 min=0.8172 max=0.8293 n=123
   Incorrect: μ=0.8216 σ=0.0028 min=0.8175 max=0.8255 n=6
   Overall  : μ=0.8229 σ=0.0018 min=0.8172 max=0.8293 n=129



MoteStrain Statistics:
   Correct  : μ=0.6081 σ=0.0297 min=0.4624 max=0.6905 n=615
   Incorrect: μ=0.6192 σ=0.0321 min=0.5487 max=0.6765 n=84
   Overall  : μ=0.6095 σ=0.0302 min=0.4624 max=0.6905 n=699

SUMMARY COMPARISON ACROSS ALL DATASETS

Datasets ranked by robustness difference (Correct - Incorrect):
Higher positive values = correctly classified samples are more robust
--------------------------------------------------------------------------------
(Correct: 0.6275, Incorrect: 0.5390)
FaceFour                 : Δ=+0.0884 
(Correct: 0.5145, Incorrect: 0.4860)
ECG200                   : Δ=+0.0285 
(Correct: 0.8433, Incorrect: 0.8281)
BirdChicken              : Δ=+0.0152 
(Correct: 0.5220, Incorrect: 0.5092)
Lightning2               : Δ=+0.0128 
(Correct: 0.8221, Incorrect: 0.8137)
BeetleFly                : Δ=+0.0085 
(Correct: 0.7097, Incorrect: 0.7017)
Wine                     : Δ=+0.0081 
(Correct: 0.7057, Incorrect: 0.6977)
SonyAIBORobotSurface1    : Δ=+0.0080 
(Correct: 0.4320

In [ ]:
# Create improved combined boxplots with better visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math
import numpy as np

print("Creating Enhanced Combined Robustness Boxplots")
print("=" * 70)

if all_dataset_data:
    # Sort datasets by accuracy for better organization
    dataset_accuracies = []
    for dataset in all_dataset_data.keys():
        accuracy_score = 0.0
        if dataset in df.columns and 'Test Accuracy' in df.index:
            try:
                acc_value = df.loc['Test Accuracy', dataset]
                if pd.notna(acc_value):
                    if isinstance(acc_value, str) and '%' in acc_value:
                        accuracy_score = float(acc_value.replace('%', '')) / 100
                    else:
                        accuracy_score = float(acc_value)
            except:
                accuracy_score = 0.0
        dataset_accuracies.append((dataset, accuracy_score))
    
    # Sort by accuracy (descending)
    dataset_accuracies.sort(key=lambda x: x[1], reverse=True)
    sorted_datasets = [d[0] for d in dataset_accuracies]
    
    # Calculate optimal layout
    n_datasets = len(sorted_datasets)
    n_cols = min(5, n_datasets)  # Max 5 columns for readability
    n_rows = math.ceil(n_datasets / n_cols)
    
    # Create enhanced subplot titles
    subplot_titles = []
    for dataset in sorted_datasets:
        # Extract accuracy and sample info
        accuracy_score = "N/A"
        if dataset in df.columns and 'Test Accuracy' in df.index:
            try:
                acc_value = df.loc['Test Accuracy', dataset]
                if pd.notna(acc_value):
                    if isinstance(acc_value, str) and '%' in acc_value:
                        accuracy_score = acc_value
                    else:
                        accuracy_score = f"{float(acc_value)*100:.1f}%"
            except:
                accuracy_score = "N/A"
        
        n_samples = all_dataset_data[dataset]['total']
        subplot_titles.append(f"<b>{dataset}</b><br>Acc: {accuracy_score} | n={n_samples}")
    
    # Create subplots with better spacing
    fig = make_subplots(
        rows=n_rows, 
        cols=n_cols,
        subplot_titles=subplot_titles,
        vertical_spacing=0.12,
        horizontal_spacing=0.06,
        specs=[[{"secondary_y": False} for _ in range(n_cols)] for _ in range(n_rows)]
    )
    
    # Enhanced color scheme with transparency
    color_map = {
        'Overall': 'rgba(31, 119, 180, 0.7)',     # Blue with transparency
        'Correct': 'rgba(44, 160, 44, 0.8)',      # Green 
        'Incorrect': 'rgba(214, 39, 40, 0.8)'     # Red
    }
    
    # Add boxplots with enhanced styling
    for idx, dataset in enumerate(sorted_datasets):
        dataset_info = all_dataset_data[dataset]
        robustness_df = dataset_info['data']
        
        # Calculate subplot position
        row = (idx // n_cols) + 1
        col = (idx % n_cols) + 1
        
        # Add traces with better styling
        for category_idx, category in enumerate(['Overall', 'Correct', 'Incorrect']):
            category_data = robustness_df[robustness_df['category'] == category]['robustness']
            
            if len(category_data) > 0:
                # Calculate statistics for hover info
                mean_val = np.mean(category_data)
                std_val = np.std(category_data)
                
                fig.add_trace(
                    go.Box(
                        y=category_data,
                        name=category,
                        marker_color=color_map[category],
                        boxpoints='outliers',
                        jitter=0.4,
                        pointpos=0,
                        showlegend=(idx == 0),
                        offsetgroup=category,
                        x=[category] * len(category_data),
                        hovertemplate=
                        f"<b>{category}</b><br>" +
                        "Robustness: %{y:.4f}<br>" +
                        f"Mean: {mean_val:.4f}<br>" +
                        f"Std: {std_val:.4f}<br>" +
                        f"Count: {len(category_data)}<br>" +
                        "<extra></extra>",
                        boxmean='sd',  # Show mean and standard deviation
                        line_width=2,
                        fillcolor=color_map[category],
                        opacity=0.8
                    ),
                    row=row, 
                    col=col
                )
    
    # Enhanced layout with better styling
    fig.update_layout(
        title={
            'text': "<b>Robustness Analysis Across All Datasets</b><br>" +
                   "<sub>Comparison of Classification Robustness (sorted by accuracy)</sub>",
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 18, 'family': 'Arial, sans-serif'},
            'y': 0.98
        },
        height=max(600, n_rows * 300),
        width=min(1600, n_cols * 300),
        boxmode='group',
        font=dict(size=11, family='Arial, sans-serif'),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="center",
            x=0.5,
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="rgba(0,0,0,0.2)",
            borderwidth=1
        ),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    
    # Update axes with better formatting
    for row in range(1, n_rows + 1):
        for col in range(1, n_cols + 1):
            fig.update_xaxes(
                title_text="Classification Type",
                row=row, col=col,
                tickangle=45,
                gridcolor='rgba(128,128,128,0.2)',
                title_font_size=10
            )
            fig.update_yaxes(
                title_text="Robustness Score",
                row=row, col=col,
                gridcolor='rgba(128,128,128,0.2)',
                range=[0, 1],  # Set consistent y-axis range
                title_font_size=10
            )
    
    # Show the enhanced plot
    fig.show()
    
    # Save in multiple formats
    fig.write_image("results/enhanced_combined_robustness_boxplots.pdf", scale=3, width=1600, height=max(600, n_rows * 300))
    fig.write_html("results/enhanced_combined_robustness_boxplots.html")
    
    print(f"\nEnhanced combined boxplot created:")
    print(f"  - Datasets: {n_datasets} (sorted by accuracy)")
    print(f"  - Layout: {n_rows} rows x {n_cols} columns")
    print(f"  - Features: Interactive hover, mean/std display, consistent scaling")
    print(f"  - Files saved:")
    print(f"    * results/enhanced_combined_robustness_boxplots.pdf")
    print(f"    * results/enhanced_combined_robustness_boxplots.html")
    
    # Print summary statistics
    print(f"\nDataset Performance Summary (sorted by accuracy):")
    print("-" * 60)
    for dataset, acc in dataset_accuracies:
        n_samples = all_dataset_data[dataset]['total']
        n_correct = all_dataset_data[dataset]['n_correct']
        print(f"{dataset:25s}: {acc*100:5.1f}% accuracy | {n_correct:3d}/{n_samples:3d} samples")
        
else:
    print("No robustness data available for combined plotting")